<a href="https://colab.research.google.com/github/uditmodi2322-bot/income/blob/main/Saving_predictor_pkl.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install scikit-learn pandas numpy gradio joblib

In [18]:
!pip install scikit-learn pandas numpy gradio joblib
import os
import glob
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import joblib
import kagglehub

# 1. Download Kaggle Dataset
path = kagglehub.dataset_download("ahammadmejbah/financial-risk-classification")
print("Path to dataset files:", path)

# 2. Locate and Load the CSV file from Kaggle
csv_files = glob.glob(os.path.join(path, "*.csv"))
if csv_files:
    kaggle_df = pd.read_csv(csv_files[0])
    print("✅ Successfully loaded downloaded Kaggle dataset!")
    print("Dataset Columns:", list(kaggle_df.columns))

# ----------------------------------------------------
# 3. Create Complete Finance Feature Dataset
# ----------------------------------------------------
np.random.seed(42)
n_samples = 5000

# Demographics & Primary Financial Inputs
age = np.random.randint(18, 65, n_samples)
monthly_income = np.random.uniform(20000, 300000, n_samples)
current_savings = np.random.uniform(10000, 2000000, n_samples)

# Assets Breakdown
num_houses = np.random.randint(0, 4, n_samples)
num_shops = np.random.randint(0, 3, n_samples)
num_farmhouses = np.random.randint(0, 2, n_samples)

# Rental Yields per property (Average per month)
house_rent = num_houses * np.random.uniform(8000, 25000, n_samples)
shop_rent = num_shops * np.random.uniform(10000, 35000, n_samples)
farmhouse_rent = num_farmhouses * np.random.uniform(15000, 50000, n_samples)
total_rental_income = house_rent + shop_rent + farmhouse_rent

# Investments Allocation
fd_amount = np.random.uniform(0, 1000000, n_samples)
stocks_amount = np.random.uniform(0, 1500000, n_samples)
monthly_sip = np.random.uniform(0, 50000, n_samples)

# Expenses & Net Monthly Savings Formula
monthly_expenses = monthly_income * np.random.uniform(0.3, 0.6, n_samples)
total_monthly_income = monthly_income + total_rental_income
net_monthly_savings = total_monthly_income - monthly_expenses - monthly_sip

# Target Calculation (5 Years)
years = 5
compounded_fd = fd_amount * ((1 + 0.065)**years)
compounded_stocks = stocks_amount * ((1 + 0.12)**years)

r_sip = 0.12 / 12
n_months = years * 12
compounded_sip = monthly_sip * (((1 + r_sip)**n_months - 1) / r_sip) * (1 + r_sip)

future_savings_5yr = (
    current_savings
    + (net_monthly_savings * n_months)
    + compounded_fd
    + compounded_stocks
    + compounded_sip
)

# Assemble DataFrame
df = pd.DataFrame({
    'age': age,
    'monthly_income': monthly_income,
    'current_savings': current_savings,
    'num_houses': num_houses,
    'num_shops': num_shops,
    'num_farmhouses': num_farmhouses,
    'total_rental_income': total_rental_income,
    'fd_amount': fd_amount,
    'stocks_amount': stocks_amount,
    'monthly_sip': monthly_sip,
    'future_savings_5yr': future_savings_5yr
})

df.to_csv('personal_finance_savings_dataset.csv', index=False)
print("✅ Dataset saved as 'personal_finance_savings_dataset.csv'")

# ----------------------------------------------------
# 4. Machine Learning Model Pipeline
# ----------------------------------------------------
X = df.drop(columns=['future_savings_5yr'])
y = df['future_savings_5yr']

# FIXED: Replaced 'test_state=42' with 'random_state=42'
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, test_size=0.2)

# Model Training
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Evaluation
y_pred = model.predict(X_test)
print(f"📊 Model Training Complete!")
print(f"-> Accuracy (R2 Score): {r2_score(y_test, y_pred):.4f}")
print(f"-> Mean Absolute Error: ₹{mean_absolute_error(y_test, y_pred):,.2f}")

# Save Model
joblib.dump(model, 'savings_predictor_model.pkl')
print("💾 Model saved as 'savings_predictor_model.pkl'")

Using Colab cache for faster access to the 'financial-risk-classification' dataset.
Path to dataset files: /kaggle/input/financial-risk-classification
✅ Successfully loaded downloaded Kaggle dataset!
Dataset Columns: ['age', 'annual_income', 'education_years', 'work_experience_years', 'credit_score', 'loan_amount', 'loan_duration_months', 'interest_rate', 'debt_to_income_ratio', 'monthly_expenses', 'savings_balance', 'investment_balance', 'number_of_dependents', 'employment_stability_years', 'housing_status_score', 'marital_status_score', 'health_risk_index', 'insurance_coverage_score', 'spending_score', 'online_activity_score', 'device_usage_hours', 'location_risk_index', 'previous_default_count', 'financial_literacy_score', 'loan_default']
✅ Dataset saved as 'personal_finance_savings_dataset.csv'
📊 Model Training Complete!
-> Accuracy (R2 Score): 0.9201
-> Mean Absolute Error: ₹823,066.80
💾 Model saved as 'savings_predictor_model.pkl'


In [23]:
import gradio as gr
import joblib
import pandas as pd

# Load trained model
model = joblib.load('savings_predictor_model.pkl')

def predict_future_finance(
    age, monthly_income, current_savings,
    num_houses, num_shops, num_farmhouses,
    avg_house_rent, avg_shop_rent, avg_farmhouse_rent,
    fd_amount, stocks_amount, monthly_sip
):
    total_rental_income = (num_houses * avg_house_rent) + (num_shops * avg_shop_rent) + (num_farmhouses * avg_farmhouse_rent)

    input_df = pd.DataFrame([{
        'age': age,
        'monthly_income': monthly_income,
        'current_savings': current_savings,
        'num_houses': num_houses,
        'num_shops': num_shops,
        'num_farmhouses': num_farmhouses,
        'total_rental_income': total_rental_income,
        'fd_amount': fd_amount,
        'stocks_amount': stocks_amount,
        'monthly_sip': monthly_sip
    }])

    predicted_savings_5yr = model.predict(input_df)[0]
    total_monthly_cashflow = monthly_income + total_rental_income
    years_to_retirement = max(0, 60 - age)

    return (
        f"₹{total_rental_income:,.2f}",
        f"₹{total_monthly_cashflow:,.2f}",
        f"₹{predicted_savings_5yr:,.2f}",
        f"{years_to_retirement} years"
    )

# Removed theme from Blocks() to fix Gradio 6 warning
with gr.Blocks() as app:
    gr.Markdown("# 💰 Personal Wealth & 5-Year Savings Predictor")

    with gr.Row():
        with gr.Column():
            gr.Markdown("### 👤 Demographic & Income")
            age = gr.Slider(18, 80, step=1, label="Age (Years)", value=28)
            monthly_income = gr.Number(label="Monthly Income / Salary (₹)", value=75000)
            current_savings = gr.Number(label="Current Liquid Savings (₹)", value=200000)

            gr.Markdown("### 🏠 Real Estate Assets")
            num_houses = gr.Slider(0, 10, step=1, label="Number of Houses", value=1)
            avg_house_rent = gr.Number(label="Avg Rent per House / Month (₹)", value=12000)

            num_shops = gr.Slider(0, 10, step=1, label="Number of Shops", value=0)
            avg_shop_rent = gr.Number(label="Avg Rent per Shop / Month (₹)", value=15000)

            num_farmhouses = gr.Slider(0, 5, step=1, label="Number of Farmhouses", value=0)
            avg_farmhouse_rent = gr.Number(label="Avg Rent per Farmhouse / Month (₹)", value=25000)

        with gr.Column():
            gr.Markdown("### 📈 Investments")
            fd_amount = gr.Number(label="Fixed Deposit (FD) Total Balance (₹)", value=100000)
            stocks_amount = gr.Number(label="Stocks Portfolio Value (₹)", value=150000)
            monthly_sip = gr.Number(label="Monthly Mutual Fund / SIP (₹)", value=10000)

            predict_btn = gr.Button("🚀 Predict Future Savings", variant="primary")

            gr.Markdown("---")
            gr.Markdown("### 📊 Predicted Output")
            out_rental = gr.Textbox(label="Total Monthly Rental Income")
            out_cashflow = gr.Textbox(label="Combined Monthly Cashflow (Salary + Rent)")
            out_prediction = gr.Textbox(label="Predicted Savings & Wealth in 5 Years")
            out_horizon = gr.Textbox(label="Years Remaining Until Retirement Age (60)")

    predict_btn.click(
        fn=predict_future_finance,
        inputs=[
            age, monthly_income, current_savings,
            num_houses, num_shops, num_farmhouses,
            avg_house_rent, avg_shop_rent, avg_farmhouse_rent,
            fd_amount, stocks_amount, monthly_sip
        ],
        outputs=[out_rental, out_cashflow, out_prediction, out_horizon]
    )

# Moved theme to launch() and removed share=True
app.launch(theme=gr.themes.Soft())

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f7a29fec5fcb1f45f2.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [13]:
#gradio
#scikit-learn==1.6.1
#pandas
#numpy
#joblib